In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/valid.csv
/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/train.csv
/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/test.csv


# Libraries

In [4]:
import pandas as pd  # Pandas for data manipulation and handling DataFrame objects
pd.set_option('display.max_colwidth',None)
import numpy as np

from sklearn.model_selection import train_test_split  # Scikit-learn's function to split data into train and test sets
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification  # HuggingFace Transformers for tokenization and model
from transformers import Trainer, TrainingArguments  # HuggingFace Trainer API for model training and fine-tuning
from datasets import Dataset  # HuggingFace datasets library to handle dataset objects and easy integration with models
import torch  # PyTorch for tensor operations and model handling
from sklearn.metrics import classification_report, confusion_matrix  # Scikit-learn for generating classification metrics

In [5]:
import nltk 
from nltk.corpus import stopwords

## dataset

In [6]:
df=pd.read_csv('/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/train.csv')
df.sample(7)

,text,intent
6135,i d like to watch bound for glory at the closest movie theatre at 21:41:08,SearchScreeningEvent
3260,give this album 4 out of 6 stars,RateBook
3566,add villotta to the metalsucks playlist playlist,AddToPlaylist
7245,add josefina pla to isabelle s guest list gamesradar playlist,AddToPlaylist
4293,play twenties from ken floyd,PlayMusic
4395,play some origa on google music,PlayMusic
245,add a kj 52 track to the te quiero playlist,AddToPlaylist


In [7]:
df.intent.value_counts()

intent
PlayMusic               1914
GetWeather              1896
BookRestaurant          1881
RateBook                1876
SearchScreeningEvent    1852
SearchCreativeWork      1847
AddToPlaylist           1818
Name: count, dtype: int64

In [8]:
df.isnull().sum()

text      0
intent    0
dtype: int64

# cleaning

In [9]:
nltk.download('stopwords')
stop_words=set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
def preprocess_text(text:str):
    text=text.lower()
    return ' '.join([word for word in text.split() if word not in stop_words])

In [11]:
text_input='stop tHe Fucnking music you are plauing for no reason'
preprocess_text(text_input)

'stop fucnking music plauing reason'

In [12]:
df['clean_text']=df['text'].apply(preprocess_text)

In [13]:
df.sample(7)

,text,intent,clean_text
6517,book a table for vermicelli at a restaurant,BookRestaurant,book table vermicelli restaurant
2132,the bone season is worth at least three points,RateBook,bone season worth least three points
11193,can you add something by faultline to jannie s strength of street knowledge playlist,AddToPlaylist,add something faultline jannie strength street knowledge playlist
10116,what is the wauseon forecast,GetWeather,wauseon forecast
2563,use spotify to play me music,PlayMusic,use spotify play music
8720,book me a reservation for fatty crew in 13 weeks in haskell,BookRestaurant,book reservation fatty crew 13 weeks haskell
3763,add a compilation of warped music ii to workout twerkout playlist,AddToPlaylist,add compilation warped music ii workout twerkout playlist


In [14]:
# Encoding Intent
unique_labels=df.intent.unique()
print(unique_labels)

label_to_id={label:i for i,label in enumerate(unique_labels)}
print(label_to_id)

df['label']=df.intent.map(label_to_id)

df.sample(7)

['PlayMusic' 'AddToPlaylist' 'RateBook' 'SearchScreeningEvent'
 'BookRestaurant' 'GetWeather' 'SearchCreativeWork']
{'PlayMusic': 0, 'AddToPlaylist': 1, 'RateBook': 2, 'SearchScreeningEvent': 3, 'BookRestaurant': 4, 'GetWeather': 5, 'SearchCreativeWork': 6}


,text,intent,clean_text,label
6219,this novel gets 1 star,RateBook,novel gets 1 star,2
10969,what will the weather be like here on dec 8 2019,GetWeather,weather like dec 8 2019,5
8318,what time is exploits of a young don juan playing,SearchScreeningEvent,time exploits young juan playing,3
1402,how cold is it in my current position,GetWeather,cold current position,5
10913,please book me a table for next week at a top-rated restaurant,BookRestaurant,please book table next week top-rated restaurant,4
2050,find the exile and the kingdom,SearchCreativeWork,find exile kingdom,6
5855,play some sad songs from the fifties,PlayMusic,play sad songs fifties,0


# Tokenization

In [15]:
# load the distilbert tokenizer
tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# calculating maximum tokenized length
max_len=max([len(tokenizer.encode(text)) for text in df['clean_text']])
print(f'max length is :{max_len}')

def tokenize_function(examples):
    tokenized_input=tokenizer(examples['clean_text'],padding='max_length',truncation=True,max_length=max_len)
    tokenized_input['labels']=examples['label']
    return tokenized_input

dataset=Dataset.from_pandas(df[['clean_text','label']])

#applying tokenization
dataset=dataset.map(tokenize_function,batched=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

max length is :28


Map:   0%|          | 0/13084 [00:00<?, ? examples/s]

In [16]:
dataset[0]

{'clean_text': 'listen westbam alumb allergic google music',
 'label': 0,
 'input_ids': [101,
  4952,
  2225,
  3676,
  2213,
  2632,
  25438,
  27395,
  8224,
  2189,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'labels': 0}

In [18]:
# Split the Pandas DataFrame (80% train, 20% validation)
from sklearn.model_selection import train_test_split
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Convert to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label']])
eval_dataset = Dataset.from_pandas(eval_df[['clean_text', 'label']])

# Map tokenization function to both datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")

Map:   0%|          | 0/10467 [00:00<?, ? examples/s]

Map:   0%|          | 0/2617 [00:00<?, ? examples/s]

Train dataset size: 10467
Eval dataset size: 2617


# Fine Tune model

In [19]:
# Initialize DistilBERT model for sequence classification
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=len(unique_labels))

# Move model to GPU if available
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [20]:
# Training arguments
training_args = TrainingArguments(  # Define the training configurations
    output_dir="./results",  # Directory to save results (model checkpoints, logs, etc.)
    # evaluation_strategy="epoch",  # Evaluate the model at the end of each epoch
    learning_rate=2e-5,  # Set learning rate for the optimizer
    per_device_train_batch_size=16,  # Batch size for training (number of examples per device)
    per_device_eval_batch_size=64,  # Batch size for evaluation
    num_train_epochs=3,  # Number of epochs to train the model
    weight_decay=0.01,  # L2 regularization to avoid overfitting
    # logging_dir="./logs",  # Directory to store training logs
    logging_steps=10,  # Log training information every 10 steps
)

# Trainer setup
trainer = Trainer(  # Initialize the Trainer with the model and training configurations
    model=model,  # Model to be trained
    args=training_args,  # Training arguments
    train_dataset=train_dataset,  # Training dataset
    eval_dataset=eval_dataset,  # Validation dataset (in practice, this should be a separate dataset)
)

# Train the model
trainer.train()  # Start training the model based on the provided training arguments

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
10,3.831458
20,3.589221
30,3.152660
40,2.606185
50,2.046578
60,1.535328
70,1.234575
80,0.940426
90,0.799302
100,0.596237


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=984, training_loss=0.3169666276260362, metrics={'train_runtime': 98.9922, 'train_samples_per_second': 317.207, 'train_steps_per_second': 9.94, 'total_flos': 227498889230424.0, 'train_loss': 0.3169666276260362, 'epoch': 3.0})

# Evaluation

In [21]:
predictions,true_labels,_=trainer.predict(dataset)

predicted_labels=predictions.argmax(axis=1)

print('Classificaiton Report :')
print(classification_report(true_labels,predicted_labels))

print('Confusion Matrix : ')
print(confusion_matrix(true_labels,predicted_labels))

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Classificaiton Report :
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1914
           1       1.00      1.00      1.00      1818
           2       1.00      1.00      1.00      1876
           3       0.99      0.98      0.99      1852
           4       1.00      1.00      1.00      1881
           5       1.00      1.00      1.00      1896
           6       0.98      0.98      0.98      1847

    accuracy                           0.99     13084
   macro avg       0.99      0.99      0.99     13084
weighted avg       0.99      0.99      0.99     13084

Confusion Matrix : 
[[1898    7    0    1    0    0    8]
 [   4 1812    2    0    0    0    0]
 [   0    0 1875    0    1    0    0]
 [   1    0    0 1816    0    0   35]
 [   0    0    0    0 1881    0    0]
 [   0    0    0    0    8 1888    0]
 [  31    1    0   10    0    1 1804]]


# Save model

In [22]:
# Save the model and tokenizer
model.save_pretrained('./saved_fine_tuned_model')
tokenizer.save_pretrained('./saved_fine_tuned_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_fine_tuned_model/tokenizer_config.json',
 './saved_fine_tuned_model/tokenizer.json')

# Inference: Prediction System

In [23]:
model=DistilBertForSequenceClassification.from_pretrained('./saved_fine_tuned_model')
tokenizer=DistilBertTokenizer.from_pretrained('./saved_fine_tuned_model')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [24]:
id_to_label={i:label for label,i in label_to_id.items()}
id_to_label

{0: 'PlayMusic',
 1: 'AddToPlaylist',
 2: 'RateBook',
 3: 'SearchScreeningEvent',
 4: 'BookRestaurant',
 5: 'GetWeather',
 6: 'SearchCreativeWork'}

In [25]:
# Funtion to make prediction
def predict(text:str,model,tokenizer,max_length=50):
    text=preprocess_text(text)
    inputs=tokenizer(text,padding='max_length',truncation=True,max_length=max_len,return_tensors='pt')
    #make predictions
    # with torch.no_grad():
    outputs=model(**inputs)
    logits=outputs.logits
    #getting predicted label(who have highest prob) 
    predicted_class_id=torch.argmax(logits,dim=-1).item()
    return predicted_class_id

In [26]:
# Define the intent labels
id_to_label = {
    0: 'getweather',
    1: 'searchcreativework',
    2: 'searchscreeningevent',
    3: 'addtoplaylist',
    4: 'bookrestaurant',
    5: 'ratebook',
    6: 'playmusic'
}

# Example messages for testing
test_messages = [
    "What's the weather like today?",
    "Find me a creative project about AI.",
    "Are there any events screening this weekend?",
    "Add this song to my playlist.",
    "I'd like to book a table for two.",
    "Rate the book I just finished reading.",
    "Play some relaxing music.",
    "Can you find a documentary on climate change?",
    "What time does the movie start tonight?",
    "Add the new album to my library."
]

# Test the model with the example messages
for message in test_messages:
    predicted_label = predict(message, model, tokenizer)
    predicted_intent = id_to_label.get(predicted_label, "Unknown Intent")
    print(f"Message: {message}")
    print(f"Predicted Label: {predicted_label}, Predicted Intent: {predicted_intent}\n")


Message: What's the weather like today?
Predicted Label: 5, Predicted Intent: ratebook

Message: Find me a creative project about AI.
Predicted Label: 6, Predicted Intent: playmusic

Message: Are there any events screening this weekend?
Predicted Label: 3, Predicted Intent: addtoplaylist

Message: Add this song to my playlist.
Predicted Label: 1, Predicted Intent: searchcreativework

Message: I'd like to book a table for two.
Predicted Label: 4, Predicted Intent: bookrestaurant

Message: Rate the book I just finished reading.
Predicted Label: 2, Predicted Intent: searchscreeningevent

Message: Play some relaxing music.
Predicted Label: 0, Predicted Intent: getweather

Message: Can you find a documentary on climate change?
Predicted Label: 6, Predicted Intent: playmusic

Message: What time does the movie start tonight?
Predicted Label: 3, Predicted Intent: addtoplaylist

Message: Add the new album to my library.
Predicted Label: 1, Predicted Intent: searchcreativework



# Download

In [29]:
import shutil
import os

# Kaggle uses /kaggle/working/ as the primary writable directory
model_dir = "/kaggle/working/saved_fine_tuned_model"

# Save the model and tokenizer
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

# Zip the model folder in the working directory
shutil.make_archive("/kaggle/working/distilbert_model_final", 'zip', model_dir)

print("Model saved and zipped successfully! You can download 'distilbert_model.zip' from the Output panel on the right side of your Kaggle editor.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved and zipped successfully! You can download 'distilbert_model.zip' from the Output panel on the right side of your Kaggle editor.


### for google colab

In [27]:
# import shutil

# # Path to save the model
# model_dir = "/content/saved_fine_tuned_model"
# model.save_pretrained(model_dir)  # Save the model
# tokenizer.save_pretrained(model_dir)  # Save the tokenizer

# # Zip the model folder
# shutil.make_archive("distilbert_model", 'zip', model_dir)

# # Download the zipped model
# from google.colab import files
# files.download("distilbert_model.zip")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>